In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from tqdm import tqdm
tqdm.pandas()

from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold

from lightgbm import LGBMClassifier
from xgboost import  XGBClassifier
from sklearn.ensemble import  RandomForestClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import f1_score


import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
pd.options.mode.chained_assignment = None  # default='warn'
#pd.set_option('display.float_format', lambda x: '%.3f' % x)
plt.rcParams["figure.figsize"] = (12, 8)
pd.set_option('display.max_columns', None)

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
os.listdir(path)

In [ ]:
# Task 1: Write your code here:

path = os.path.join(path, 'Q1_data.csv')

train_df = pd.read_csv(path)


In [ ]:
# Task 2: Write your code here:
train_df.head()

In [ ]:
# Task 3: Write your code here:
train_df.info()

In [ ]:
# Task 4: Write your code here:
train_df.describe()

In [ ]:
# Task 5: Write your code here:

# Price distribution (target variable)
plt.figure(figsize=(10, 5))
plt.hist(train_df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery_Time Distribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
train_df.drop(columns=['Order_ID'])

In [ ]:
# Task 2: Write your code here:

def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(train_df)

In [ ]:
categorical_cols = train_df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

In [ ]:
cols = ['Distance_km', 'Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type', 'Preparation_Time_min', 'Courier_Experience_yrs', 'Delivery_Time']
df_clean = train_df[cols].copy()

# Drop rows where target () or key features are missing - can't predict without them
print(f"Before: {df_clean.shape}")
df_clean = df_clean.dropna(subset=['Delivery_Time', 'Courier_Experience_yrs'])
print(f"After dropping missing Delivery_Time/Courier_Experience_yrs: {df_clean.shape}")

In [ ]:
# Fill categorical columns with 'unknown' - missing likely means "not specified"
for col in ['Weather', 'Traffic_Level', 'Time_of_Day',  'Vehicle_Type']:
    #df_clean[col] = df_clean[col].fillna(df_clean['col'].mode()[0])
    df_clean[col] = df_clean[col].fillna('unknown')



print("Missing values remaining:", df_clean.isnull().sum().sum())

In [ ]:
# Task 3: Write your code here:

def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import OneHotEncoder

# Encode categorical columns - converts text to integers
categorical_cols = ['Weather', 'Traffic_Level', 'Time_of_Day',  'Vehicle_Type']
print('data before encoding:\n', categorical_cols) #show before encoding

for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

df_clean.head()

In [ ]:
# Task 5: Write your code here:

from sklearn.preprocessing import StandardScaler

features = df_clean.columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df_clean[features] = scaler.fit_transform(df_clean[features])
df_clean.head()

In [ ]:
# Task 6: Write your code here:

In [ ]:
def check_target_imbalance(df, target_column):
    print("Target Distribution:")
    print(df[target_column].value_counts(normalize=True))
    sns.countplot(x=df[target_column])
    plt.title("Target Distribution")
    plt.show()

check_target_imbalance(df_clean, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:

X = df_clean.drop("Delivery_Time", axis=1).astype(float)
y = df_clean['Delivery_Time'].astype(float)

In [ ]:
# Calculate the majority class baseline
majority_class = y.value_counts().idxmax()
baseline_pred = [majority_class] * len(y)

# Evaluate the baseline
baseline_f1 = f1_score(y, baseline_pred, average='weighted')
print(f"Baseline F1-Score (majority class): {baseline_f1:.4f}")


In [ ]:
from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor


In [ ]:
from sklearn.metrics import mean_absolute_error

# Initialize KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Initialize RandomForest model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)

# Store MAE for each fold
mae_scores = []

# Perform KFold
for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Train the model
    rf_model.fit(X_train, y_train)

    # Make predictions
    y_pred = rf_model.predict(X_test)

    # Calculate MAE
    mae = mean_absolute_error(y_test, y_pred)
    mae_scores.append(mae)

# Output average MAE
print(f'Average MAE across folds: {np.mean(mae_scores)}')


In [ ]:
# Task 1: Write your code here:

# Retrieve CatBoost feature importances and sort them
# Retrieve CatBoost feature importances and sort them
catboost_model = rf_model
catboost_importance = list(zip(X.columns, catboost_model.feature_importances_))
sorted_catboost_importance = sorted(catboost_importance, key=lambda x: x[1], reverse=True)

# Extract features and their importances
features, importances = zip(*sorted_catboost_importance)

# Plot feature importances
plt.figure(figsize=(18, 14))
plt.barh(features, importances, color='orange')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.title('RandomForestRegressor Feature Importance')
plt.gca().invert_yaxis()  # Invert y-axis to show the most important features at the top
plt.show()

In [ ]:
!pip install catboost

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here:

# Initialize models
rf_model = RandomForestRegressor()
catboost_model = CatBoostRegressor(silent=True)

# KFold setup
kf = KFold(n_splits=5)
mae_scores = []

for train_index, test_index in kf.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Train both models
    rf_model.fit(X_train, y_train)
    catboost_model.fit(X_train, y_train)

    # Make predictions
    rf_predictions = rf_model.predict(X_test)
    catboost_predictions = catboost_model.predict(X_test)

    # Average predictions
    averaged_predictions = (rf_predictions + catboost_predictions) / 2

    # Calculate MAE
    mae = mean_absolute_error(y_test, averaged_predictions)
    mae_scores.append(mae)

# Output the average MAE across all folds
print("Average MAE across all folds:", np.mean(mae_scores))